# Reaktive Agenten mit OPC UA-Integration (Mesa 3.0.x)

Dieses Notebook verbindet zwei Welten:

- die **agentenbasierte Simulation** mit Mesa 3.0.x,
- die **industrielle Kommunikation** über OPC UA (asyncua).

## Lernziele

Nach diesem Notebook können Sie:

1. erklären, warum Mesa-Simulation und OPC UA-Kommunikation **entkoppelt** betrieben werden,
2. einen **gemeinsamen Datenpuffer** zwischen synchronem Simulationscode und asynchronem Netzwerkcode einsetzen,
3. Mesa-Agenten so erweitern, dass sie **Zustandsdaten in einen OPC UA-Server schreiben**,
4. den OPC UA HOT-Monitor **parallel** zur laufenden Simulation betreiben.

## Übergeordnete Architektur

```
┌─────────────────────────────────────────────────────────┐
│                   Jupyter-Prozess                        │
│                                                          │
│  Mesa-Simulation (synchron)                              │
│  ┌─────────────────────────────┐                         │
│  │ MachineAgent.step()         │                         │
│  │   → schreibt in            │                         │
│  │     opcua_write_buffer      │──────┐                  │
│  └─────────────────────────────┘      │                  │
│                                       ▼                  │
│  asyncio Event-Loop (parallel)  ┌─────────────────────┐  │
│  ┌──────────────────────┐       │ opcua_write_buffer   │  │
│  │ opcua_writer()       │◄──────│ {"M01": {Temp, State,│  │
│  │   → write_value()    │       │          Busy}, ...} │  │
│  └──────────────────────┘       └─────────────────────┘  │
│  ┌──────────────────────┐                                 │
│  │ subscription_hot_    │                                 │
│  │ monitor()            │                                 │
│  │   → liest vom Server │                                 │
│  │   → setzt RepairNeeded│                                │
│  └──────────────────────┘                                 │
│                    │                                      │
└────────────────────┼──────────────────────────────────────┘
                     │ OPC UA (TCP)
                     ▼
        ┌─────────────────────────┐
        │  OPC UA Factory-Server  │
        │  M01/Temperature        │
        │  M01/State              │
        │  M01/Busy               │
        │  M01_RepairNeeded       │
        │  ...                    │
        └─────────────────────────┘
```

**Warum ein Puffer statt direkter OPC UA-Aufrufe im Agenten?**

`MachineAgent.step()` läuft synchron im Mesa-Scheduler. OPC UA-Schreiboperationen
sind `async` und können nicht direkt aus synchronem Code aufgerufen werden.
Der Puffer (`opcua_write_buffer`) entkoppelt beide Welten sauber:
Mesa schreibt Werte hinein, ein asyncio-Task liest sie heraus und überträgt
sie an den Server – ohne gegenseitige Blockierung.


## 1. Imports und Ausgabe-Widget

Zuerst richten wir das `Output`-Widget ein und leiten `print` global um,
damit OPC UA-Ausgaben aus asyncio-Callbacks sichtbar werden.


In [ ]:
import asyncio
import builtins
import ipywidgets as widgets
from IPython.display import display

# Der OPC UA HOT-Monitor-Client muss als OPC_Hot_Monitor_Client_de.py
# im selben Verzeichnis liegen.
import OPC_Hot_Monitor_Client_de as hot_client

# ── Output-Widget für OPC UA Meldungen ────────────────────────────────────────
# Alle print()-Aufrufe – auch aus asyncio-Callbacks – landen hier.
opcua_out = widgets.Output(layout=widgets.Layout(
    height="150px",
    overflow_y="auto",
    border="1px solid gray",
))

_original_print = builtins.print
def _patched_print(*args, **kwargs):
    with opcua_out:
        _original_print(*args, **kwargs)
builtins.print = _patched_print

print("Mesa + OPC UA Umgebungs-Imports bereit.")


## 2. Gemeinsamer Datenpuffer

Der `opcua_write_buffer` ist das zentrale Bindeglied zwischen Mesa und OPC UA.

- **Mesa schreibt** nach jedem `step()` die aktuellen Agentenwerte hinein.
- **Der asyncio Writer-Task liest** den Puffer periodisch und überträgt die Werte
  an den OPC UA-Server.

Da Mesa und der Writer-Task im selben Python-Prozess laufen und der asyncio
Event-Loop kooperativ ist (kein echtes Multithreading), sind keine Locks
erforderlich – es gibt keine Race Conditions.

```
MachineAgent.step()          asyncio Writer-Task
      │                             │
      │  opcua_write_buffer         │
      │  ┌─────────────────────┐    │
      └─►│ "M01": {42.3, "WARM"│──► │ write_value() → OPC UA Server
         │ "M02": {71.1, "HOT" │    │
         └─────────────────────┘    │
```


In [ ]:
# Gemeinsamer Datenpuffer: Maschinenname → aktuelle Zustandswerte
# Wird von MachineAgent.step() beschrieben und vom opcua_writer() gelesen.
opcua_write_buffer: dict = {}

# Puffer für RepairNeeded-Flags die zurückgesetzt werden sollen
# key = Maschinenname, value = False (RepairNeeded zurücksetzen)
opcua_repair_buffer: dict = {}

## 3. Mesa-Agenten mit OPC UA-Anbindung

Gegenüber der ursprünglichen Version gibt es zwei Erweiterungen:

### 3.1 MachineAgent: `machine_name` und Puffer-Update

Jeder `MachineAgent` bekommt einen festen **OPC UA-Maschinennamen** (z.B. `"M01"`),
der dem Knoten auf dem Server entspricht. Am Ende jedes `step()` schreibt er
seine aktuellen Werte in den `opcua_write_buffer`.

**Warum am Ende von `step()` und nicht sofort auf den Server?**
OPC UA-Schreiboperationen sind `async` – sie können nicht direkt aus synchronem
Mesa-Code aufgerufen werden. Der Puffer ist die saubere Entkopplung.

### 3.2 FactoryModelExtended: Maschinen nummerieren

Damit jeder Agent einen eindeutigen Servernamen hat, werden die Maschinen beim
Erstellen durchnummeriert: `M01`, `M02`, ...


In [ ]:
from mesa import Agent
import random


class MachineAgent(Agent):
    """Reaktiver Maschinenagent mit OPC UA-Anbindung über gemeinsamen Datenpuffer.

    Erweiterungen gegenüber der Basisversion:
    - machine_name: eindeutiger Name des OPC UA-Knotens (z.B. 'M01')
    - step() schreibt Temperatur, Zustand und Busy-Flag in opcua_write_buffer
    """

    def __init__(self, model, threshold=70, machine_name=None):
        super().__init__(model)
        self.temperature = 20.0
        self.threshold = threshold
        self.state = "OK"
        self.busy = False
        self.machine_name = machine_name  # ← OPC UA-Knotenname, z.B. "M01"

    def sense_temperature(self):
        if self.state != "HOT":
            self.temperature += random.uniform(-1, 2)

    def decide(self):
        if self.temperature > self.threshold:
            self.state = "HOT"
        elif self.temperature > 0.75 * self.threshold:
            self.state = "WARM"
        elif self.temperature < 20:
            self.state = "COOL"
        else:
            self.state = "OK"

    def act(self):
        pass  # Abkühlung durch MaintenanceAgent

    def step(self):
        self.sense_temperature()
        self.decide()
        self.act()
        # ── OPC UA: Zustand in Puffer schreiben ───────────────────────────
        # Der asyncio Writer-Task überträgt den Puffer periodisch an den Server.
        if self.machine_name:
            opcua_write_buffer[self.machine_name] = {
                "Temperature": round(self.temperature, 2),
                "State":       self.state,
                "Busy":        self.busy,
            }


class MaintenanceAgent(Agent):
    """Wandert durch das Gitter und repariert HOT-Maschinen."""

    def step(self):
        neighbours = self.model.grid.get_neighborhood(
            self.pos, moore=True, include_center=True
        )
        self.model.grid.move_agent(self, random.choice(neighbours))

        for agent in self.model.grid.get_cell_list_contents([self.pos]):
            if isinstance(agent, MachineAgent) and agent.state == "HOT":
                agent.temperature = agent.threshold - 40
                agent.state = "OK"
                agent.busy = False
                # ── OPC UA: RepairNeeded zurücksetzen ─────────────────────
                if agent.machine_name:
                    opcua_repair_buffer[agent.machine_name] = False


class OrderAgent(Agent):
    """Produktionsauftrag: sucht eine freie Maschine, belegt sie für 'duration' Schritte."""

    def __init__(self, model, duration=10):
        super().__init__(model)
        self.duration = duration
        self.assigned_machine = None

    def step(self):
        # Fall 1: noch keine Maschine → suchen und zuweisen
        if self.assigned_machine is None:
            neighbours = self.model.grid.get_neighborhood(
                self.pos, moore=True, include_center=True
            )
            self.model.grid.move_agent(self, random.choice(neighbours))

            for agent in self.model.grid.get_cell_list_contents([self.pos]):
                if isinstance(agent, MachineAgent) and agent.state != "HOT" and not agent.busy:
                    agent.busy = True
                    agent.temperature += 5
                    self.assigned_machine = agent
                    break

        # Fall 2: Maschine belegt → Dauer herunterzählen
        else:
            self.duration -= 1
            for agent in self.model.grid.get_cell_list_contents([self.pos]):
                if isinstance(agent, MachineAgent) and agent.state != "HOT":
                    agent.temperature += 5
            if self.duration <= 0:
                self.assigned_machine.busy = False
                self.model.grid.remove_agent(self)
                self.remove()


In [ ]:
from mesa import Model
from mesa.space import MultiGrid
from mesa.datacollection import DataCollector


def count_hot_machines(model):
    return sum(1 for a in model.agents if isinstance(a, MachineAgent) and a.state == "HOT")

def count_warm_machines(model):
    return sum(1 for a in model.agents if isinstance(a, MachineAgent) and a.state == "WARM")

def count_orders(model):
    return sum(1 for a in model.agents if isinstance(a, OrderAgent))


class FactoryModelExtended(Model):
    """Erweitertes Fabrikmodell mit Maschinen-, Wartungs- und Auftragsagenten.

    Erweiterung: Maschinen erhalten eindeutige Namen (M01, M02, ...),
    die den OPC UA-Knoten auf dem Server entsprechen.
    """

    def __init__(self, width=10, height=10, density=0.3, threshold=70,
                 n_maintenance=2, n_orders=5, seed=None):
        super().__init__(seed=seed)
        self.width = width
        self.height = height
        self.grid = MultiGrid(width, height, torus=False)

        # Auftragsagenten zuerst platzieren (niedrigere unique_ids)
        for _ in range(n_orders):
            pos = (self.random.randrange(width), self.random.randrange(height))
            self.grid.place_agent(OrderAgent(self), pos)

        # Maschinenagenten mit OPC UA-Namen nummerieren
        machine_counter = 1
        for x in range(width):
            for y in range(height):
                if self.random.random() < density:
                    name = f"M{machine_counter:02d}"
                    self.grid.place_agent(
                        MachineAgent(self, threshold=threshold, machine_name=name), (x, y)
                    )
                    machine_counter += 1

        for _ in range(n_maintenance):
            pos = (self.random.randrange(width), self.random.randrange(height))
            self.grid.place_agent(MaintenanceAgent(self), pos)

        self.datacollector = DataCollector(
            model_reporters={
                "HotMachines":  count_hot_machines,
                "WarmMachines": count_warm_machines,
                "Auftraege":    count_orders,
            }
        )
        self.datacollector.collect(self)

    def step(self):
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)


## 4. Visualisierung


In [ ]:
from mesa.visualization import SolaraViz, make_plot_component, make_space_component

DENSITY      = 0.2
THRESHOLD    = 70
N_MAINTENANCE = 2
N_ORDERS     = 30


def agent_portrayal_ext(agent):
    if isinstance(agent, MachineAgent):
        return {
            "color": {
                "HOT":  "red",
                "WARM": "orange",
                "COOL": "skyblue",
                "OK":   "green",
            }.get(agent.state, "gray"),
            "size": 600, "marker": "s", "alpha": 0.7,
        }
    if isinstance(agent, MaintenanceAgent):
        return {"color": "cyan",  "size": 300, "marker": "o", "alpha": 0.95}
    if isinstance(agent, OrderAgent):
        return {"color": "black", "size": 200, "marker": "s", "alpha": 0.85}
    return {}


def make_post_process_ext(model):
    def post_process(ax):
        ax.figure.set_size_inches(6, 6)
        legend = ax.get_legend()
        if legend:
            legend.remove()
        for txt in ax.texts[:]:
            txt.remove()
        for agent in model.agents:
            x, y = agent.pos
            if isinstance(agent, MachineAgent):
                # OPC UA-Name und Temperatur anzeigen
                label = f"{agent.machine_name}\n{agent.temperature:.1f}°C" if agent.machine_name else f"{agent.temperature:.1f}°C"
                ax.text(x, y, label,
                        ha="center", va="center",
                        fontsize=5, color="white", fontweight="bold")
            elif isinstance(agent, MaintenanceAgent):
                ax.text(x, y, "W",
                        ha="center", va="center",
                        fontsize=9, color="black", fontweight="bold")
            elif isinstance(agent, OrderAgent):
                ax.text(x, y, str(agent.unique_id),
                        ha="center", va="center",
                        fontsize=6, color="white")
        ax.grid(True, alpha=0.3)
    return post_process


model_ext = FactoryModelExtended(
    width=10, height=10, density=DENSITY, threshold=THRESHOLD,
    n_maintenance=N_MAINTENANCE, n_orders=N_ORDERS,
)

space_ext = make_space_component(
    agent_portrayal_ext,
    post_process=make_post_process_ext(model_ext),
)
plot_ext = make_plot_component(
    {"HotMachines": "red", "WarmMachines": "orange", "Auftraege": "black"}
)


## 5. OPC UA HOT-Monitor (Lesen vom Server)

Der HOT-Monitor liest Temperaturen vom OPC UA-Server und setzt
`Mxx_RepairNeeded = True`, sobald eine Maschine HOT wird.

Er läuft als asyncio-Task parallel zur Simulation – beide teilen sich
den Jupyter Event-Loop, blockieren sich aber nicht gegenseitig.


In [ ]:
hot_monitor_task = None

def start_hot_monitor_in_background(
    mode: str = "subscription",
    runtime_seconds: float = 300.0,
    interval: float = 1.0,
    publishing_interval_ms: int = 500,
):
    """OPC UA HOT-Monitor als asyncio-Hintergrundtask starten."""
    global hot_monitor_task
    loop = asyncio.get_event_loop()

    if mode == "polling":
        coro = hot_client.polling_hot_monitor(
            runtime_seconds=runtime_seconds,
            poll_interval=interval,
        )
    else:
        coro = hot_client.subscription_hot_monitor(
            runtime_seconds=runtime_seconds,
            publishing_interval_ms=publishing_interval_ms,
        )

    hot_monitor_task = loop.create_task(coro)
    print(f"HOT-Monitor gestartet ({mode}-Modus). Task:", hot_monitor_task)


def cancel_hot_monitor():
    """Laufenden HOT-Monitor-Task abbrechen."""
    global hot_monitor_task
    if hot_monitor_task is not None and not hot_monitor_task.done():
        hot_monitor_task.cancel()
        print("HOT-Monitor-Task abgebrochen.")
    else:
        print("Kein laufender HOT-Monitor-Task gefunden.")


## 6. OPC UA Writer-Task (Schreiben auf den Server)

Dieser Task ist das Herzstück der neuen Kopplung. Er:

1. **verbindet sich einmalig** mit dem OPC UA-Server,
2. **löst alle Maschinen-Nodes einmalig auf** und speichert sie im `node_cache`
   (das ist deutlich effizienter als bei jedem Schreibvorgang `get_child()` aufzurufen),
3. **liest periodisch** den `opcua_write_buffer` und schreibt die Werte auf den Server.

### Warum Node-Caching?

`get_child()` ist ein Netzwerkaufruf – bei jedem Schreibvorgang würde das
die Latenz unnötig erhöhen. Durch einmaliges Auflösen der Nodes beim Start
reduzieren wir jeden Schreibzyklus auf reine `write_value()`-Aufrufe.

### Adressraum-Mapping

Der OPC UA-Server legt Maschinen unter folgendem Pfad an:

```
Objects / Factory / Machines / M01 / Temperature
                                   / State
                                   / Busy
```

Der Namespace-Index `nsidx` wird dynamisch vom Server abgefragt –
keine hartcodierten Indizes.


In [ ]:
from asyncua import Client

OPC_UA_URL    = "opc.tcp://localhost:4840/freeopcua/server/"
OPC_UA_NS_URI = "http://ostfalia.de/ipt/factory"

opcua_writer_task = None  # globaler Handle zum Abbrechen


async def opcua_writer(runtime_seconds: float = 300.0, interval: float = 1.0):
    """Liest opcua_write_buffer und schreibt Temperatur, State und Busy auf den OPC UA-Server.

    Ablauf:
        1. Verbindung aufbauen und Namespace-Index ermitteln.
        2. Alle Maschinen-Nodes einmalig auflösen und cachen.
        3. Schreibschleife: Puffer lesen → write_value() → warten.

    Parameter
    ----------
    runtime_seconds : float
        Laufzeit des Writers in Sekunden.
    interval : float
        Pause zwischen zwei Schreibzyklen in Sekunden.
    """
    async with Client(url=OPC_UA_URL) as client:
        nsidx = await client.get_namespace_index(OPC_UA_NS_URI)
        print(f"OPC UA Writer verbunden. Namespace-Index: {nsidx}")

        # ── Schritt 1: Alle Maschinen-Nodes einmalig auflösen ─────────────
        # Wir versuchen M01…M50; nicht vorhandene Nodes werden übersprungen.
        node_cache: dict = {}
        for i in range(1, 51):
            name = f"M{i:02d}"
            base = f"0:Objects/{nsidx}:Factory/{nsidx}:Machines/{nsidx}:{name}"
            try:
                node_cache[name] = {
                    "Temperature": await client.nodes.root.get_child(
                        f"{base}/{nsidx}:Temperature"),
                    "State":       await client.nodes.root.get_child(
                        f"{base}/{nsidx}:State"),
                    "Busy":        await client.nodes.root.get_child(
                        f"{base}/{nsidx}:Busy"),
                }
            except Exception:
                break  # Keine weiteren Maschinen auf dem Server

        print(f"OPC UA Writer: {len(node_cache)} Maschinen-Nodes gecacht ({', '.join(node_cache)}).")

        # ── RepairNeeded-Nodes cachen ────────────────────────────────
        # Pfad: Objects/Factory/Maintenance/Jobs/Mxx_RepairNeeded
        repair_cache: dict = {}
        jobs_base = f"0:Objects/{nsidx}:Factory/{nsidx}:Maintenance/{nsidx}:Jobs"
        for name in node_cache:
            try:
                repair_cache[name] = await client.nodes.root.get_child(
                    f"{jobs_base}/{nsidx}:{name}_RepairNeeded"
                )
            except Exception as e:
                print(f"RepairNeeded-Node für {name} nicht gefunden: {e}")

        print(f"OPC UA Writer: {len(repair_cache)} RepairNeeded-Nodes gecacht.")

        # ── Alle RepairNeeded-Flags beim Start auf False setzen ───────
        for name, node in repair_cache.items():
            try:
                await node.write_value(False)
            except Exception as e:
                print(f"Initialisierungsfehler {name}_RepairNeeded: {e}")
        print(f"Alle {len(repair_cache)} RepairNeeded-Flags initialisiert (→ False).")
    

        # ── Schritt 2: Schreibschleife ─────────────────────────────────────
        deadline = asyncio.get_event_loop().time() + runtime_seconds
        while asyncio.get_event_loop().time() < deadline:
            for machine_name, values in list(opcua_write_buffer.items()):
                if machine_name not in node_cache:
                    continue  # Maschine existiert nicht auf dem Server
                nodes = node_cache[machine_name]
                try:
                    await nodes["Temperature"].write_value(float(values["Temperature"]))
                    await nodes["State"].write_value(str(values["State"]))
                    await nodes["Busy"].write_value(bool(values["Busy"]))
                except Exception as e:
                    print(f"Schreibfehler {machine_name}: {e}")

            await asyncio.sleep(interval)

            # ── NEU: RepairNeeded-Flags zurücksetzen ──────────────────────
            for machine_name, value in list(opcua_repair_buffer.items()):
                if machine_name not in repair_cache:
                    continue
                try:
                    await repair_cache[machine_name].write_value(value)
                    del opcua_repair_buffer[machine_name]  # ← einmalig schreiben, dann löschen
                    print(f"{machine_name}_RepairNeeded → {value}")
                except Exception as e:
                    print(f"RepairNeeded-Schreibfehler {machine_name}: {e}")
            
        print("OPC UA Writer beendet.")


def start_opcua_writer(runtime_seconds: float = 300.0, interval: float = 1.0):
    """OPC UA Writer als asyncio-Hintergrundtask starten."""
    global opcua_writer_task
    loop = asyncio.get_event_loop()
    opcua_writer_task = loop.create_task(
        opcua_writer(runtime_seconds=runtime_seconds, interval=interval)
    )
    print(f"OPC UA Writer gestartet (Intervall: {interval}s). Task:", opcua_writer_task)


def cancel_opcua_writer():
    """Laufenden Writer-Task abbrechen."""
    global opcua_writer_task
    if opcua_writer_task is not None and not opcua_writer_task.done():
        opcua_writer_task.cancel()
        print("OPC UA Writer-Task abgebrochen.")
    else:
        print("Kein laufender Writer-Task gefunden.")


## 7. Simulation, HOT-Monitor und Writer gemeinsam starten

Jetzt laufen drei Komponenten parallel im selben Jupyter-Prozess:

| Komponente | Art | Aufgabe |
|---|---|---|
| `SolaraViz` | synchron (Hauptthread) | Mesa-Simulation visualisieren |
| `subscription_hot_monitor` | asyncio-Task | Temperaturen vom Server lesen, RepairNeeded setzen |
| `opcua_writer` | asyncio-Task | Puffer auf den Server schreiben |

**Reihenfolge:** Zuerst Writer und Monitor starten, dann `page_ext` als
letzte Zellenausgabe – sonst blockiert SolaraViz den Event-Loop bevor
die Tasks registriert sind.


In [ ]:
# OPC UA Writer starten (schreibt Mesa-Zustände auf den Server)
start_opcua_writer(runtime_seconds=300, interval=1.0)

# HOT-Monitor starten (liest vom Server, setzt RepairNeeded-Flags)
start_hot_monitor_in_background(mode="subscription", runtime_seconds=300)

# Mesa-Simulation starten
page_ext = SolaraViz(
    model_ext,
    components=[space_ext, plot_ext],
    name="Erweitertes Fabrikmodell mit OPC UA",
    play_interval=200,
)


### OPC UA Ausgaben anzeigen


In [ ]:
display(opcua_out)

In [ ]:
page_ext

## 8. Tasks beenden

Mit den folgenden Zellen können die Hintergrundtasks jederzeit manuell
abgebrochen werden – z.B. um Parameter zu ändern und neu zu starten.


In [ ]:
# cancel_hot_monitor()
# cancel_opcua_writer()


## 9. Zusammenfassung und Erweiterungsmöglichkeiten

### Was wurde umgesetzt?

- **MachineAgent** schreibt nach jedem `step()` Temperatur, Zustand und Busy-Flag
  in den `opcua_write_buffer`.
- **`opcua_writer()`** überträgt den Puffer periodisch auf den OPC UA-Server –
  vollständig entkoppelt vom Mesa-Scheduler.
- **HOT-Monitor** liest Temperaturen vom Server und setzt `RepairNeeded`-Flags.

### Architekturprinzip: Lese/Schreibe-Entkopplung

```
Mesa (synchron)          Puffer          OPC UA (async)
─────────────────        ──────          ──────────────
step() → schreiben  ──►  dict    ──►  write_value()
                                  ◄──  read_value() (HOT-Monitor)
```

### Mögliche nächste Schritte

1. **RepairNeeded-Flags lesen**: `MaintenanceAgent` könnte `Mxx_RepairNeeded`
   vom Server lesen und gezielt zu überhitzten Maschinen navigieren.

2. **Aufträge über OPC UA koordinieren**: `OrderAgent` könnte freie Maschinen
   über den Server-Adressraum suchen statt über das lokale Grid.

3. **Schreibintervall anpassen**: Für reale Anlagen reicht oft `interval=2.0` –
   das reduziert Netzwerklast erheblich.

4. **Writer in einen Mesa-Agenten kapseln**: Ein `OpcUaWriterAgent` könnte
   den Puffer im `step()` leeren – dann ist der Writer vollständig in das
   Agentensystem integriert.
